# Assignment Sesi 28 Tugas 1
Nama: Faraday Barr Fatahillah

Dataset: [movies_metadata.csv](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv) 

**Tugas 1**

Pada tugas ini gunakanlah dataset film yang dapat diunduh dari Kaggle [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv). Gunakan file `movies_metadata.csv` sebagai sumber data.Dari data tersebut kerjakanlah beberapa perintah berikut:
1. Buatlah keyword Search menggunakan BM25 dan TF-IDF untuk mencari film berkaitan dengan:
    - robot
    - love
    - adventure
2. Buatlah Semantic Search dengan Vector Database (Pinecone dan ChromaDB) untuk mencari film berkaitan dengan:
    - Adventure movie with characters named Judy and Peter
    - Movies with scenes in New York City
    - movie about poet Arthur Rimbaud and Paul Verlaine relationship

In [1]:
# import needed library
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import numpy as np
import chromadb
import google.generativeai as genai
import time
from pinecone import Pinecone, ServerlessSpec

c:\Users\bobe\anaconda3\envs\bootcamp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\bobe\AppData\Local\Temp\ipykernel_23672\24560260.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
df = pd.read_csv('movies_metadata.csv')

df.head()

C:\Users\bobe\AppData\Local\Temp\ipykernel_23672\681536703.py:1: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('movies_metadata.csv')


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [4]:
df.isna().sum().sort_values(ascending=False)

belongs_to_collection    40972
homepage                 37684
tagline                  25054
overview                   954
poster_path                386
runtime                    263
status                      87
release_date                87
imdb_id                     17
original_language           11
vote_average                 6
vote_count                   6
title                        6
video                        6
spoken_languages             6
revenue                      6
popularity                   5
production_countries         3
production_companies         3
genres                       0
id                           0
adult                        0
budget                       0
original_title               0
dtype: int64

In [5]:
df = df[['title', 'overview']].dropna()

df.isna().sum().sort_values(ascending=False)

title       0
overview    0
dtype: int64

## Keyword Search

### TF-IDF Search

In [6]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix  = tfidf.fit_transform(df['overview'])

In [7]:
def format_output(df, top_indices):
    return {
            i: {
                "title": df['title'].iloc[i],
                "overview": df['overview'].iloc[i]
            }
            for i in top_indices
        }




def tfidf_search(query, top_n=10):
    query_vec = tfidf.transform([query])
    cosine_sim = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = cosine_sim.argsort()[-top_n:][::-1]
    return format_output(df, top_indices)

In [8]:
df_robot = pd.DataFrame(tfidf_search("robot")).T.reset_index(drop=True)
df_love = pd.DataFrame(tfidf_search("love")).T.reset_index(drop=True)
df_adventure = pd.DataFrame(tfidf_search("adventure")).T.reset_index(drop=True)


In [9]:
df_robot

,title,overview
0,"OMG, I'm a Robot!",A sensitive guy finds out he's... a robot.
1,Cody the Robosapien,"At Kinetech Labs, an inventor named Allan Toph..."
2,Making Mr. Right,A reclusive scientist builds a robot that look...
3,Robot Stories,"Four stories including: ""My Robot Baby,"" in wh..."
4,Eve of Destruction,Eve is a military robot made to look exactly l...
5,Robotrix,A mad scientist transfers his mind to a wicked...
6,BURN·E,What lengths will a robot undergo to do his jo...
7,Bicentennial Man,"Richard Martin buys a gift, a new NDR-114 robo..."
8,Blinky™,"A story about a boy, his robot and the consequ..."
9,ABE,A short film about a robot programmed for love...


In [10]:
df_love

,title,overview
0,In Lieu of Flowers,Most love stories are about losing love or fin...
1,Nigdy w życiu!,Journey to find true love.
2,E Aí... Comeu?,The real first comedy about love.
3,"Crouching Tiger, Hidden Dragon: Sword of Destiny","A story of lost love, young love, a legendary ..."
4,For a Handful of Kisses,A girl. A boy. A love story. But also about dr...
5,A Bela e o Paparazzo,Love on the front page
6,Julietta,A dramatic teenage love story set against the ...
7,ABE,A short film about a robot programmed for love...
8,Backstairs,A crippled mailman is in love with a maid who ...
9,Winter Cherries,A story about a difficult love between the two...


In [11]:
df_adventure

,title,overview
0,Dinotopia: Quest for the Ruby Sunstone,An Orphaned Boy sets out in search of adventur...
1,Pernicious,It was supposed to be an adventure of a lifeti...
2,White Wilderness,A fabulous new adventure in exciting entertain...
3,Camille,A twisted honeymoon adventure about a young co...
4,Tex,Coming-of-age adventure about two teenage brot...
5,Приключения Шерлока Холмса и доктора Ватсона: ...,The Twentieth Century Approaches is a 1986 Sov...
6,A Perfect Christmas List,"As a last wish, a recently hospitalized grandm..."
7,3 Idiotas,A group of friends embark on an adventure to f...
8,The Challenge,The Challenge is an action adventure/adventure...
9,Taxi No. 9 2 11: Nau Do Gyarah,A cabbie and businessman both in need of big m...


### BM25

In [12]:
tokenized_corpus = [doc.split(" ") for doc in df['overview']]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_n=10):
    tokenized_query = query.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[-top_n:][::-1]
    return format_output(df, top_indices)

In [13]:
df_robot_bm = pd.DataFrame(tfidf_search("robot")).T.reset_index(drop=True)
df_love_bm = pd.DataFrame(tfidf_search("love")).T.reset_index(drop=True)
df_adventure_bm = pd.DataFrame(tfidf_search("adventure")).T.reset_index(drop=True)


In [14]:
df_robot_bm

,title,overview
0,"OMG, I'm a Robot!",A sensitive guy finds out he's... a robot.
1,Cody the Robosapien,"At Kinetech Labs, an inventor named Allan Toph..."
2,Making Mr. Right,A reclusive scientist builds a robot that look...
3,Robot Stories,"Four stories including: ""My Robot Baby,"" in wh..."
4,Eve of Destruction,Eve is a military robot made to look exactly l...
5,Robotrix,A mad scientist transfers his mind to a wicked...
6,BURN·E,What lengths will a robot undergo to do his jo...
7,Bicentennial Man,"Richard Martin buys a gift, a new NDR-114 robo..."
8,Blinky™,"A story about a boy, his robot and the consequ..."
9,ABE,A short film about a robot programmed for love...


In [15]:
df_love_bm

,title,overview
0,In Lieu of Flowers,Most love stories are about losing love or fin...
1,Nigdy w życiu!,Journey to find true love.
2,E Aí... Comeu?,The real first comedy about love.
3,"Crouching Tiger, Hidden Dragon: Sword of Destiny","A story of lost love, young love, a legendary ..."
4,For a Handful of Kisses,A girl. A boy. A love story. But also about dr...
5,A Bela e o Paparazzo,Love on the front page
6,Julietta,A dramatic teenage love story set against the ...
7,ABE,A short film about a robot programmed for love...
8,Backstairs,A crippled mailman is in love with a maid who ...
9,Winter Cherries,A story about a difficult love between the two...


In [16]:
df_adventure_bm

,title,overview
0,Dinotopia: Quest for the Ruby Sunstone,An Orphaned Boy sets out in search of adventur...
1,Pernicious,It was supposed to be an adventure of a lifeti...
2,White Wilderness,A fabulous new adventure in exciting entertain...
3,Camille,A twisted honeymoon adventure about a young co...
4,Tex,Coming-of-age adventure about two teenage brot...
5,Приключения Шерлока Холмса и доктора Ватсона: ...,The Twentieth Century Approaches is a 1986 Sov...
6,A Perfect Christmas List,"As a last wish, a recently hospitalized grandm..."
7,3 Idiotas,A group of friends embark on an adventure to f...
8,The Challenge,The Challenge is an action adventure/adventure...
9,Taxi No. 9 2 11: Nau Do Gyarah,A cabbie and businessman both in need of big m...


## Semantic Search

### ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Akhirnya pakai model lokal saja, sudah tidak kuat 4 akun Gemini AI Studio untuk encode 45k data, sudah kena rate limit semua. Kalau pakai model lokal, bisa encode semua data sekaligus (batch) dan jauh lebih cepat.
embeddings = model.encode(df['overview'].tolist(), show_progress_bar=True)
df['embedding'] = list(embeddings)

Batches: 100%|██████████| 1391/1391 [03:39<00:00,  6.33it/s]


In [22]:
client = chromadb.Client()
collection = client.create_collection(name="movies")

In [23]:
for i, row in df.iterrows():
    collection.add(
        documents=[row['overview']],
        embeddings=[row['embedding']],
        metadatas=[{"title": row['title']}],
        ids=[str(i)]
    )

In [24]:
def chroma_search(query, top_n=5):
    query_embedding = model.encode(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_n
    )
    
    titles = results['metadatas'][0]
    docs = results['documents'][0]
    for i, (meta, doc) in enumerate(zip(titles, docs)):
        print(f"{i+1}. {meta['title']}")
        print(f"   {doc[:150]}...")
        print()
    
    return results


In [25]:
print("=== Query 1 ===")
chroma_search("Adventure movie with characters named Judy and Peter")

=== Query 1 ===
1. Tom and Jerry & The Wizard of Oz
   They're off to see the Wizard, the wonderful Wizard of Oz! Tom and Jerry soar over the rainbow and travel down the yellow brick road in this all-anima...

2. Raising Victor Vargas
   The film follows Victor, a Lower East Side teenager, as he deals with his eccentric family, including his strict grandmother, his bratty sister, and a...

3. Peter & the Wolf
   An animated retelling set to Prokofiev's suite. Peter is a slight lad, solitary, locked out of the woods by his protective grandfather...

4. The Wind in the Willows
   Kenneth Grahame's literary classic about an enchanting world along the Riverbank has delighted readers for nearly a century. Now, this enduring belove...

5. Jumanji
   When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's b...



{'ids': [['31259', '6130', '15268', '27165', '1']],
 'embeddings': None,
 'documents': [['They\'re off to see the Wizard, the wonderful Wizard of Oz! Tom and Jerry soar over the rainbow and travel down the yellow brick road in this all-animated retelling of the classic tale. You\'ll see your favorite characters: Dorothy, Toto, the Scarecrow, the Tin Man, the Lion, the Wicked Witch of the West, the Wizard, the Munchkins and more. You\'ll hear many of your favorite songs, including "Over The Rainbow". And you\'ll laugh at the antics of your favorite cat and mouse as they get twisted up in a twister, go paw-to-paw against flying monkeys and storm the Wicked Witch\'s castle in a heroic attempt to get Dorothy and Toto (and themselves) safely back to Kansas. After all, there\'s no place like home.',
   'The film follows Victor, a Lower East Side teenager, as he deals with his eccentric family, including his strict grandmother, his bratty sister, and a younger brother who completely idolizes 

In [26]:
print("=== Query 2 ===")
chroma_search("Movies with scenes in New York City")

=== Query 2 ===
1. No More Excuses
   A bizarre portrait of the New York singles scene....

2. Kurt Metzger: White Precious
   Standup special filmed at the Gramercy Theatre in New York.....

3. Broadway Damage
   Romantic comedy about aspiring writers in NY....

4. Greenwich Village: Music That Defined a Generation
   Explores the music scene in Greenwich Village, New York in the 60's and early 70's. The film highlights some of the finest singer/songwriters of the d...

5. A Most Violent Year
   A thriller set in New York City during the winter of 1981, statistically one of the most violent years in the city's history, and centered on a the li...



{'ids': [['39584', '42760', '3014', '22436', '25466']],
 'embeddings': None,
 'documents': [['A bizarre portrait of the New York singles scene.',
   'Standup special filmed at the Gramercy Theatre in New York..',
   'Romantic comedy about aspiring writers in NY.',
   "Explores the music scene in Greenwich Village, New York in the 60's and early 70's. The film highlights some of the finest singer/songwriters of the day.",
   "A thriller set in New York City during the winter of 1981, statistically one of the most violent years in the city's history, and centered on a the lives of an immigrant and his family trying to expand their business and capitalize on opportunities as the rampant violence, decay, and corruption of the day drag them in and threaten to destroy all they have built."]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'title': 'No More Excuses'},
   {'title': 'Kurt Metzger: White Precious'},
   {'title': 'Broadway Dam

In [27]:
print("=== Query 3 ===")
chroma_search("movie about poet Arthur Rimbaud and Paul Verlaine relationship")

=== Query 3 ===
1. Total Eclipse
   Young, wild poet Arthur Rimbaud and his mentor Paul Verlaine engage in a fierce, forbidden romance while feeling the effects of a hellish artistic lif...

2. Harlan: In the Shadow of Jew Süss
   Though almost forgotten today, Veit Harlan was one of Nazi Germany's most notorious filmmakers. His most perfidious film was the treacherous anti-Semi...

3. Monsieur Verdoux
   The film is about an unemployed banker, Henri Verdoux, and his sociopathic methods of attaining income. While being both loyal and competent in his wo...

4. Portrait Werner Herzog
   is an autobiographical short film by Werner Herzog made in 1986. Herzog tells stories about his life and career.  The film contains excerpts and comme...

5. Wild Reeds
   Set in rural France during the end of the Algerian war, and immersed in the music of the sixties, this film follows four students in friendship and lo...



{'ids': [['199', '16423', '3510', '17950', '869']],
 'embeddings': None,
 'documents': [['Young, wild poet Arthur Rimbaud and his mentor Paul Verlaine engage in a fierce, forbidden romance while feeling the effects of a hellish artistic lifestyle.',
   "Though almost forgotten today, Veit Harlan was one of Nazi Germany's most notorious filmmakers. His most perfidious film was the treacherous anti-Semitic propaganda film Jud Süß - required viewing for all SS members. An unrepentant and blindly obsessive craftsman, no figure - save for Leni Riefenstahl - is as closely associated with the cinema of the Holocaust years. (Harlan's epic Kolberg was the basis for Inglourious Basterds's pivotal film-within-a-film Stolz Der Nation.) This documentary is an eye-opening examination of World War II film history as well as the story of a German family from the Third Reich to the present; one that is marked by reckoning, denial and liberation.",
   'The film is about an unemployed banker, Henri Verdo

### Pinecone

In [38]:
pc = Pinecone(api_key='pcsk_3tkfqG_JfxwytHV5n1eYCyGFrQ7kyPP76oNQJEraqqGFteC63Bsnhb3XghYfx79UekCrVJ')

index_name = "movies-semantic-384"

existing_indexes = pc.list_indexes().names()
if index_name in existing_indexes:
    index_info = pc.describe_index(index_name)
    if index_info.dimension != 384:
        print(f"Index exists with dimension {index_info.dimension}, deleting and recreating...")
        pc.delete_index(index_name)
        existing_indexes = []

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)


In [39]:
batch_size = 100
vectors = []

for i, row in df.iterrows():
    vectors.append({
        "id": str(i),
        "values": row['embedding'].tolist(),
        "metadata": {
            "title": row['title'],
            "overview": row['overview'][:500]
        }
    })
    
    if len(vectors) == batch_size:
        index.upsert(vectors=vectors)
        vectors = []

if vectors:
    index.upsert(vectors=vectors)


In [40]:
def pinecone_search(query, top_n=5):
    query_embedding = model.encode(query).tolist()
    
    results = index.query(
        vector=query_embedding,
        top_k=top_n,
        include_metadata=True
    )
    
    for i, match in enumerate(results['matches']):
        print(f"{i+1}. {match['metadata']['title']} (score: {match['score']:.4f})")
        print(f"   {match['metadata']['overview'][:150]}...")
        print()
    
    return results


In [41]:
print("=== Query 1 ===")
pinecone_search("Adventure movie with characters named Judy and Peter")

=== Query 1 ===
1. Tom and Jerry & The Wizard of Oz (score: 0.5152)
   They're off to see the Wizard, the wonderful Wizard of Oz! Tom and Jerry soar over the rainbow and travel down the yellow brick road in this all-anima...

2. Raising Victor Vargas (score: 0.5079)
   The film follows Victor, a Lower East Side teenager, as he deals with his eccentric family, including his strict grandmother, his bratty sister, and a...

3. Peter & the Wolf (score: 0.5059)
   An animated retelling set to Prokofiev's suite. Peter is a slight lad, solitary, locked out of the woods by his protective grandfather...

4. The Wind in the Willows (score: 0.4961)
   Kenneth Grahame's literary classic about an enchanting world along the Riverbank has delighted readers for nearly a century. Now, this enduring belove...

5. Jumanji (score: 0.4891)
   When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's b...



QueryResponse(matches=[ScoredVector(id='31259', score=0.515213, values=[], metadata={'overview': 'They\'re off to see the Wizard, the wonderful Wizard of Oz! Tom and Jerry soar over the rainbow and travel down the yellow brick road in this all-animated retelling of the classic tale. You\'ll see your favorite characters: Dorothy, Toto, the Scarecrow, the Tin Man, the Lion, the Wicked Witch of the West, the Wizard, the Munchkins and more. You\'ll hear many of your favorite songs, including "Over The Rainbow". And you\'ll laugh at the antics of your favorite cat and mouse as they get twisted up in ', 'title': 'Tom and Jerry & The Wizard of Oz'}), ScoredVector(id='6130', score=0.507883132, values=[], metadata={'overview': 'The film follows Victor, a Lower East Side teenager, as he deals with his eccentric family, including his strict grandmother, his bratty sister, and a younger brother who completely idolizes him. Along the way he tries to win the affections of Judy, who is very careful a

In [42]:
print("=== Query 2 ===")
pinecone_search("Movies with scenes in New York City")

=== Query 2 ===
1. No More Excuses (score: 0.6188)
   A bizarre portrait of the New York singles scene....

2. Kurt Metzger: White Precious (score: 0.6121)
   Standup special filmed at the Gramercy Theatre in New York.....

3. Broadway Damage (score: 0.5838)
   Romantic comedy about aspiring writers in NY....

4. Greenwich Village: Music That Defined a Generation (score: 0.5783)
   Explores the music scene in Greenwich Village, New York in the 60's and early 70's. The film highlights some of the finest singer/songwriters of the d...

5. A Most Violent Year (score: 0.5718)
   A thriller set in New York City during the winter of 1981, statistically one of the most violent years in the city's history, and centered on a the li...



QueryResponse(matches=[ScoredVector(id='39584', score=0.618826807, values=[], metadata={'overview': 'A bizarre portrait of the New York singles scene.', 'title': 'No More Excuses'}), ScoredVector(id='42760', score=0.612108171, values=[], metadata={'overview': 'Standup special filmed at the Gramercy Theatre in New York..', 'title': 'Kurt Metzger: White Precious'}), ScoredVector(id='3014', score=0.583841324, values=[], metadata={'overview': 'Romantic comedy about aspiring writers in NY.', 'title': 'Broadway Damage'}), ScoredVector(id='22436', score=0.578265131, values=[], metadata={'overview': "Explores the music scene in Greenwich Village, New York in the 60's and early 70's. The film highlights some of the finest singer/songwriters of the day.", 'title': 'Greenwich Village: Music That Defined a Generation'}), ScoredVector(id='25466', score=0.571763039, values=[], metadata={'overview': "A thriller set in New York City during the winter of 1981, statistically one of the most violent year

In [43]:
print("=== Query 3 ===")
pinecone_search("movie about poet Arthur Rimbaud and Paul Verlaine relationship")

=== Query 3 ===
1. Total Eclipse (score: 0.7381)
   Young, wild poet Arthur Rimbaud and his mentor Paul Verlaine engage in a fierce, forbidden romance while feeling the effects of a hellish artistic lif...

2. Harlan: In the Shadow of Jew Süss (score: 0.5107)
   Though almost forgotten today, Veit Harlan was one of Nazi Germany's most notorious filmmakers. His most perfidious film was the treacherous anti-Semi...

3. Monsieur Verdoux (score: 0.4998)
   The film is about an unemployed banker, Henri Verdoux, and his sociopathic methods of attaining income. While being both loyal and competent in his wo...

4. Portrait Werner Herzog (score: 0.4757)
   is an autobiographical short film by Werner Herzog made in 1986. Herzog tells stories about his life and career.  The film contains excerpts and comme...

5. Wild Reeds (score: 0.4753)
   Set in rural France during the end of the Algerian war, and immersed in the music of the sixties, this film follows four students in friendship and lo...



QueryResponse(matches=[ScoredVector(id='199', score=0.73812443, values=[], metadata={'overview': 'Young, wild poet Arthur Rimbaud and his mentor Paul Verlaine engage in a fierce, forbidden romance while feeling the effects of a hellish artistic lifestyle.', 'title': 'Total Eclipse'}), ScoredVector(id='16423', score=0.510739863, values=[], metadata={'overview': "Though almost forgotten today, Veit Harlan was one of Nazi Germany's most notorious filmmakers. His most perfidious film was the treacherous anti-Semitic propaganda film Jud Süß - required viewing for all SS members. An unrepentant and blindly obsessive craftsman, no figure - save for Leni Riefenstahl - is as closely associated with the cinema of the Holocaust years. (Harlan's epic Kolberg was the basis for Inglourious Basterds's pivotal film-within-a-film Stolz Der Nation.) This documentary is ", 'title': 'Harlan: In the Shadow of Jew Süss'}), ScoredVector(id='3510', score=0.499806881, values=[], metadata={'overview': 'The film